# 09 - Compare

Plan section 39: cross-seed stability, architecture equivalence, and the final comparison table.

Reproducibility contract (plan section 40): every session begins by
determining its configuration, locating and validating prior state,
and continues from there; it ends by saving metrics and a summary JSON.
Run inside a Kaggle session (or locally) after checking out this
repository, with the repository root as the working directory.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / 'src' / 'act').is_dir(), (
    f'run this notebook with the repository root as the working directory; '
    f'cwd was {REPO!r}'
)
sys.path.insert(0, str(REPO / 'src'))
print('python', sys.version.split()[0], '| repo', REPO)


# Final comparison (plan section 39)

The scientific analysis notebook: a multi-seed gate study reduced
to its cross-seed stability summary (mean, standard deviation,
range of the paired deltas) and the strict architecture-hash
equivalence check, rendered as the compact markdown table shape
the final report uses.


In [ ]:
from act.evolution.experiment import ExperimentConfig
from act.evolution.study import run_gate_study

study = run_gate_study(
    ExperimentConfig(
        steps=48, arms='random,adaptive', device='cpu',
    ),
    seeds=(1, 2, 3, 42),
    treatment='adaptive',
)
assert study.matched_mutation_budget
print('wins:', study.adaptive_wins, '/', len(study.seeds),
      '| mean delta:', round(study.mean_adaptive_delta, 6))

In [ ]:
# Cross-seed stability of the paired deltas.
from act.evaluation.stability import (
    architecture_equivalence,
    summarize_stability,
)

deltas = tuple(
    next(arm for arm in experiment.arms if arm.mode == 'adaptive').validation_loss
    - next(arm for arm in experiment.arms if arm.mode == 'random').validation_loss
    for experiment in study.experiments
)
summary = summarize_stability(deltas)
assert summary.minimum <= summary.mean <= summary.maximum
print('delta mean/stddev/min/max:',
      [round(value, 6) for value in (
          summary.mean, summary.stddev, summary.minimum, summary.maximum
      )])

In [ ]:
# Architecture equivalence: identical hashes when (and only when) the
# architectures coincide - whichever way this seed's mutations land.
hashes = tuple(arm.architecture_hash for arm in study.experiments[0].arms)
assert len(hashes) == 2
same = architecture_equivalence(hashes)
assert same == (hashes[0] == hashes[1])
assert not architecture_equivalence((hashes[0], '0' * 16))
print('arm hashes:', [value[:8] for value in hashes],
      '| equivalent:', same)

In [ ]:
# The final table shape: one row per arm, objective statistics only.
rows = [
    '| metric | value |',
    '|---|---|',
    f"| wins | {study.adaptive_wins}/{len(study.seeds)} |",
    f"| mean delta | {study.mean_adaptive_delta:.6f} |",
    f"| delta stddev | {study.delta_stddev:.6f} |",
    f"| binomial p | {study.gate_p_value:.6g} |",
    f"| gate passed | {study.gate_passed} |",
]
print('\n'.join(rows))

In [ ]:
import json
from pathlib import Path

record_dir = Path('artifacts/notebook-records')
record_dir.mkdir(parents=True, exist_ok=True)
payload = {'notebook': '09_compare',
 'wins': study.adaptive_wins,
 'seeds': list(study.seeds),
 'mean_delta': study.mean_adaptive_delta,
 'delta_stddev': study.delta_stddev,
 'gate_p_value': study.gate_p_value,
 'gate_passed': study.gate_passed,
 'delta_summary': {
     'mean': summary.mean,
     'stddev': summary.stddev,
     'minimum': summary.minimum,
     'maximum': summary.maximum,
 },
 'arms_architecture_equivalent': same}
record = json.dumps(payload, sort_keys=True, indent=2) + '\n'
target = record_dir / '09-compare.json'
existing = target.read_text() if target.exists() else None
if existing != record:
    target.write_text(record)
print('record:', target)